# Subreddit groups v5 (v4 sample, pre-accusation subreddits)

Supersedes `subreddit_groups_v4_rule_based.ipynb`: banned and private are **no longer groups** (2026-10-07 decision). Their
subreddits are placed in topic groups, and `later_banned` / `private` are kept as yes/no flags for robustness checks
(they are 2026 status, i.e. hindsight, so not main-model variables).

Group per subreddit, in priority order:
1. `profile` — `u_<name>` user-profile subreddits (own vs other's profile is decided per action);
2. `adult` — live subreddit labelled NSFW by Reddit (`over18`), or banned/private subreddit where ≥ 90% of the sample's
   archived posts there are NSFW (that rule agreed with Reddit's label on 98.5% of live subreddits);
3. live subreddits: labels from `subreddit_llm_labels_v4.csv` (agents read title/bio/sidebar per
   `subreddit_classification_guide.md`);
4. other banned/private subreddits: labels from `subreddit_llm_labels_nobio_v5.csv` (agents read the name plus the
   sample's archived titles, link domains and comment text per `subreddit_classification_guide_without_bio.md`).

Groups: profile, adult, crypto, politics, karma_farming, general, news, sales, location, other.
Outputs (never overwritten): `subreddit_review_output/subreddit_groups_v5.parquet` and `subreddit_groups_v5_review.xlsx`.

In [ ]:
# Libraries, folders, the group list, and the NSFW-post share above which a no-metadata subreddit counts as adult.
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
GROUPS = ['profile', 'adult', 'crypto', 'politics', 'karma_farming', 'general', 'news', 'sales', 'location', 'other']
ADULT_NSFW_SHARE = 0.9

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# PRAW metadata, agent labels made with the live bio, agent labels made without a bio (for banned/private
# subreddits), and the share of NSFW posts per subreddit in the archived sample.
meta = pd.read_parquet(os.path.join(OUT_DIR, 'subreddit_praw_meta_v4.parquet'))
live_labels = pd.read_csv(os.path.join(OUT_DIR, 'subreddit_llm_labels_v4.csv')).drop(columns='batch')
nobio_labels = pd.read_csv(os.path.join(OUT_DIR, 'subreddit_llm_labels_nobio_v5.csv')).drop(columns='batch')
posts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_posts.parquet'), columns=['subreddit', 'over_18'])
nsfw_share = posts.assign(nsfw=posts['over_18'].astype(str).eq('True')).groupby('subreddit')['nsfw'].mean()

# Banned and private are now flags, not groups.
g = meta[['subreddit', 'subreddit_id', 'status', 'subreddit_type', 'over18', 'title', 'public_description']].copy()
is_profile = g['subreddit'].str.startswith('u_')
no_meta = g['status'].str.startswith('banned') & ~is_profile           # PRAW returned no metadata
g['later_banned'] = no_meta & g['subreddit_type'].isin(['public', 'restricted', 'archived'])
g['private'] = no_meta & g['subreddit_type'].eq('private')
g['archived_nsfw_share'] = g['subreddit'].map(nsfw_share)
live_adult = g['status'].eq('ok') & ~is_profile & g['over18'].fillna(False).astype(bool)
nometa_adult = no_meta & g['archived_nsfw_share'].ge(ADULT_NSFW_SHARE)

# Rule-based groups: profile, then adult (live NSFW flag, or at least 90% NSFW posts when there is no metadata).
g['group'] = np.select([is_profile, live_adult | nometa_adult], ['profile', 'adult'], default='')
g['group_source'] = np.select([is_profile, live_adult, nometa_adult], ['rule_profile', 'rule_reddit_nsfw', 'rule_archived_nsfw'], default='')
# All other subreddits take their agent label; checks that rules and labels do not overlap.
lab = pd.concat([live_labels.rename(columns={'source': 'evidence'}).assign(label_set='live_bio'),
                 nobio_labels.assign(label_set='nobio')], ignore_index=True).rename(columns={'group': 'agent_group'})
assert lab['subreddit'].is_unique
g = g.merge(lab, on='subreddit', how='left', validate='one_to_one')
need = g['group'].eq('')
assert g.loc[need, 'label_set'].notna().all(), 'a subreddit needing a label has none'
assert g.loc[~need, 'label_set'].isna().all(), 'a rule-based subreddit also has an agent label'
g.loc[need, 'group_source'] = 'agent_' + g.loc[need, 'label_set'] + '_' + g.loc[need, 'evidence']
g.loc[need, 'group'] = g.loc[need, 'agent_group']
assert set(g['group']) <= set(GROUPS)
# Groups against the later-banned and private flags.
print(pd.crosstab(g['group'], g['later_banned'].map({True: 'later banned', False: ''}) + g['private'].map({True: 'private', False: ''})).to_string())

In [ ]:
# Every post and comment the sample made before its first (edit-adjusted) accusation, tagged with its subreddit's group.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
acts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'created_utc', 'subreddit'])
    acts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])])
acts = pd.concat(acts)
g = g.join(acts.groupby('subreddit').agg(actions=('outer_key', 'size'), accounts=('outer_key', 'nunique')), on='subreddit')
acts['group'] = acts['subreddit'].map(g.set_index('subreddit')['group'])
assert acts['group'].notna().all()
# Per group: subreddits, actions and accounts, and the share of all pre-accusation actions.
summary = pd.DataFrame({'subreddits': g['group'].value_counts(), 'actions': acts['group'].value_counts(),
                        'accounts': acts.groupby('group')['outer_key'].nunique()}).reindex(GROUPS)
summary['pct_actions'] = (100 * summary['actions'] / summary['actions'].sum()).round(1)
summary

In [ ]:
# Save the groups, and a review sheet sorted by group and use.
save_new(g, 'subreddit_groups_v5.parquet')
review = g.sort_values(['group', 'actions'], ascending=[True, False])[
    ['subreddit', 'group', 'group_source', 'confidence', 'reason', 'later_banned', 'private', 'archived_nsfw_share',
     'actions', 'accounts', 'title', 'public_description']]
save_new(review, 'subreddit_groups_v5_review.xlsx')